# ADVIS - Model Retraining
### Run each cell one by one, top to bottom.
### First: Runtime menu → Change runtime type → T4 GPU

In [ ]:
# CELL 1 — Check GPU is active
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('NO GPU — Go to Runtime > Change runtime type > T4 GPU, then re-run')
!nvidia-smi

In [ ]:
# CELL 2 — Install Detectron2 (takes 3-5 minutes)
!pip install -q torch torchvision
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print('Done!')

In [ ]:
# CELL 3 — Mount Google Drive and unzip dataset
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

DRIVE_ZIP     = '/content/drive/MyDrive/ADVIS/dataset.zip'
LOCAL_DATASET = '/content/Car Damage V5.v1i.coco-segmentation'

if not os.path.exists(LOCAL_DATASET):
    print('Unzipping dataset... (1-2 minutes)')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall('/content')
    print('Unzip complete!')
else:
    print('Dataset already extracted.')

train_count = len([f for f in os.listdir(LOCAL_DATASET+'/train') if f.endswith('.jpg')])
print(f'Train images found: {train_count}')
print('Ready!')

In [ ]:
# CELL 4 — Fix annotations and register datasets
import os, json
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.utils.logger import setup_logger
setup_logger()

BASE_DIR    = '/content'
DATASET_DIR = BASE_DIR + '/Car Damage V5.v1i.coco-segmentation'
OUTPUT_DIR  = BASE_DIR + '/TRAIN_OUTPUT'
os.makedirs(OUTPUT_DIR + '/annotations', exist_ok=True)

def fix_annotations(json_path):
    with open(json_path) as f:
        data = json.load(f)
    valid_cats = {c['id']: c for c in data['categories'] if c['id'] != 0}
    sorted_ids = sorted(valid_cats.keys())
    id_remap   = {old: new for new, old in enumerate(sorted_ids)}
    new_cats = []
    for old_id in sorted_ids:
        cat = valid_cats[old_id].copy()
        cat['id'] = id_remap[old_id]
        new_cats.append(cat)
    new_anns = []
    for ann in data['annotations']:
        if ann['category_id'] in id_remap:
            a = ann.copy()
            a['category_id'] = id_remap[ann['category_id']]  
            new_anns.append(a)
    data['categories']  = new_cats
    data['annotations'] = new_anns
    return data

def save_fixed(json_path, out_path):
    data = fix_annotations(json_path)
    with open(out_path, 'w') as f:
        json.dump(data, f)
    print(f'  Saved: {out_path}  ({len(data["annotations"])} annotations)')
    return out_path

print('Fixing annotations...')
fixed_train = save_fixed(DATASET_DIR+'/train/_annotations.coco.json', OUTPUT_DIR+'/annotations/train_fixed.json')
fixed_val   = save_fixed(DATASET_DIR+'/valid/_annotations.coco.json', OUTPUT_DIR+'/annotations/val_fixed.json')
fixed_test  = save_fixed(DATASET_DIR+'/test/_annotations.coco.json',  OUTPUT_DIR+'/annotations/test_fixed.json')

CLASS_NAMES = ['dent', 'glass_break', 'scratch', 'smash']
for name in ['advis_train', 'advis_val', 'advis_test']:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

register_coco_instances('advis_train', {}, fixed_train, DATASET_DIR+'/train')
register_coco_instances('advis_val',   {}, fixed_val,   DATASET_DIR+'/valid')
register_coco_instances('advis_test',  {}, fixed_test,  DATASET_DIR+'/test')
for n in ['advis_train', 'advis_val', 'advis_test']:
    MetadataCatalog.get(n).thing_classes = CLASS_NAMES

print('Datasets registered!')

In [ ]:
# CELL 5 — Configure and START TRAINING (~2 hours on T4 GPU)
import torch
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator
from detectron2.data import DatasetMapper, build_detection_train_loader
import detectron2.data.transforms as T

TRAIN_IMAGES = 4210
NUM_CLASSES  = 4
BATCH_SIZE   = 2
EPOCHS       = 10          # 10 epochs = ~2 hours on T4, safe for Colab free tier
MAX_ITER     = (TRAIN_IMAGES // BATCH_SIZE) * EPOCHS   # = 21,050 iterations
MODEL_CONFIG = 'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(MODEL_CONFIG))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(MODEL_CONFIG)
cfg.DATASETS.TRAIN = ('advis_train',)
cfg.DATASETS.TEST  = ('advis_val',)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH  = BATCH_SIZE
cfg.SOLVER.BASE_LR        = 0.001
cfg.SOLVER.MAX_ITER       = MAX_ITER
cfg.SOLVER.STEPS          = (int(MAX_ITER * 0.65), int(MAX_ITER * 0.85))
cfg.SOLVER.GAMMA          = 0.1
cfg.SOLVER.WARMUP_ITERS   = 500
cfg.SOLVER.WARMUP_FACTOR  = 1.0 / 1000
cfg.SOLVER.WEIGHT_DECAY   = 0.0001
cfg.SOLVER.CLIP_GRADIENTS.ENABLED    = True
cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 1.0
cfg.SOLVER.CHECKPOINT_PERIOD = 5000
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256
cfg.MODEL.ROI_HEADS.NUM_CLASSES          = NUM_CLASSES
cfg.INPUT.MIN_SIZE_TRAIN = (640, 672, 704, 736, 768, 800)
cfg.INPUT.MAX_SIZE_TRAIN = 1333
cfg.INPUT.MIN_SIZE_TEST  = 800
cfg.INPUT.MAX_SIZE_TEST  = 1333
cfg.INPUT.RANDOM_FLIP    = 'horizontal'
cfg.TEST.EVAL_PERIOD = 5000
cfg.OUTPUT_DIR = OUTPUT_DIR

class ADVISTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, 'coco_eval')
        return COCOEvaluator(dataset_name, cfg, False, output_dir=output_folder)
    @classmethod
    def build_train_loader(cls, cfg):
        augmentations = [
            T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
            T.RandomBrightness(0.7, 1.3),
            T.RandomContrast(0.7, 1.3),
            T.RandomSaturation(0.7, 1.3),
            T.ResizeShortestEdge(
                short_edge_length=cfg.INPUT.MIN_SIZE_TRAIN,
                max_size=cfg.INPUT.MAX_SIZE_TRAIN,
                sample_style='choice',
            ),
        ]
        mapper = DatasetMapper(cfg, is_train=True, augmentations=augmentations)
        return build_detection_train_loader(cfg, mapper=mapper)

print(f'Training: {MAX_ITER} iterations ({EPOCHS} epochs) — approx 2 hours')
print(f'LR={cfg.SOLVER.BASE_LR} | ROI proposals={cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE}')

trainer = ADVISTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()
print('Training complete!')

In [ ]:
# CELL 6 — Evaluate accuracy on test set
from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

cfg.MODEL.WEIGHTS = OUTPUT_DIR + '/model_final.pth'
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.05
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST   = 0.35
cfg.DATASETS.TEST = ('advis_test',)

predictor  = DefaultPredictor(cfg)
evaluator  = COCOEvaluator('advis_test', cfg, False, output_dir=OUTPUT_DIR+'/eval')
val_loader = build_detection_test_loader(cfg, 'advis_test')
results    = inference_on_dataset(predictor.model, val_loader, evaluator)

bbox_ap50 = float(results.get('bbox', {}).get('AP50', 0.0))
segm_ap50 = float(results.get('segm', {}).get('AP50', 0.0))
print(f'bbox AP50 : {bbox_ap50:.1f}%')
print(f'segm AP50 : {segm_ap50:.1f}%')
if bbox_ap50 >= 70:
    print('Good model! Proceed to save it.')
else:
    print('Score is low — save anyway and we can improve further.')

In [ ]:
# CELL 7 — Save model to Google Drive
import shutil
DRIVE_SAVE = '/content/drive/MyDrive/ADVIS/model_final_new.pth'
shutil.copy(OUTPUT_DIR + '/model_final.pth', DRIVE_SAVE)
print('Model saved to Google Drive at:', DRIVE_SAVE)
print()
print('NEXT STEPS:')
print('1. Download model_final_new.pth from Google Drive > ADVIS folder')
print('2. Rename it to model_final.pth')
print('3. Replace TRAIN/model_final.pth in your project folder')
print('4. Restart the app: python -m streamlit run app.py')